In [1]:
import requests
import hashlib
import pyarrow
import pandas as pd
import json
import numpy as np
import pickle
from pathlib import Path
from transformers import AutoTokenizer

/Users/quinnhinde-schuster/PycharmProjects/Accenture-1O-contract-review-challenge/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Task 13 — Evaluation & Results

This notebook explores the performance of the simple classifier on the test data (data/cuad/test.json).

to begin we need to prep the test data to run through the simple classifier by normalizing and chunking it.

-produce macro-F1 and micro metrics

-produce category-level precision, recall, and F1

-comparison to performance of validation split results



Load the data, the official raw test data

In [2]:
# Load the authorized official test file
from utils import load_file_from_github
raw_test = load_file_from_github("data/cuad/test.json")

Now that we have the raw test data, we need to normalize it. This has been done to the training data in normalize_data.ipynb, so the same format will be used here.

In [3]:
from utils import normalize_cuad

# Extract the list of contracts from the JSON dictionary
raw_test_data = raw_test["data"]

# Normalize the official test data
normalized_test_data = normalize_cuad(raw_test_data)

# Extract the five DataFrames
test_contracts = normalized_test_data["contracts"]
test_documents = normalized_test_data["documents"]
test_categories = normalized_test_data["categories"]
test_annotation_sets = normalized_test_data["annotation_sets"]
test_spans = normalized_test_data["spans"]

# Check the results
print("Contracts:", test_contracts.shape)
print("Documents:", test_documents.shape)
print("Categories:", test_categories.shape)
print("Annotation sets:", test_annotation_sets.shape)
print("Spans:", test_spans.shape)


Contracts: (102, 2)
Documents: (102, 4)
Categories: (41, 3)
Annotation sets: (4182, 4)
Spans: (2643, 6)


Save the normalized test data outputs

In [4]:
#Save to directory
output_dir = Path("task13_outputs/official_test_normalized")
output_dir.mkdir(parents=True, exist_ok=True)

test_contracts.to_parquet(output_dir / "contracts.parquet", index=False)
test_documents.to_parquet(output_dir / "documents.parquet", index=False)
test_categories.to_parquet(output_dir / "categories.parquet", index=False)
test_annotation_sets.to_parquet(output_dir / "annotation_sets.parquet", index=False)
test_spans.to_parquet(output_dir / "spans.parquet", index=False)

print("\nSaved normalized official test data")


Saved normalized official test data


Now that our data is normalized, the next thing we need to do is chunk it! This process has been defined already in task 10. Like normalizing, we will use code that has been produced already to chunk the raw test data.

In [5]:
from chunking_helper import create_paragraph_chunks

MODEL_ID = "distilroberta-base"

OUTPUT_DIR = Path("task13_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    use_fast=True
)

official_test_chunks = create_paragraph_chunks(
    test_documents,
    tokenizer=tokenizer,
    chunk_size=512,
    long_paragraph_overlap=64,
    method_name="paragraph_based"
)

official_test_chunks = (
    official_test_chunks
    .sort_values(["contract_id", "chunk_number"])
    .reset_index(drop=True)
)

print("Official test documents:", len(test_documents))
print("Official test chunks:", len(official_test_chunks))

display(official_test_chunks.head())


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (573 > 512). Running this sequence through the model will result in indexing errors


Official test documents: 102
Official test chunks: 2984


,method,contract_id,context_group_id,chunk_id,chunk_number,char_start,char_end,num_tokens
0,paragraph_based,contract_0001,context_11ef4095ae,contract_0001__chunk_0000,0,0,2063,468
1,paragraph_based,contract_0001,context_11ef4095ae,contract_0001__chunk_0001,1,2069,3587,346
2,paragraph_based,contract_0001,context_11ef4095ae,contract_0001__chunk_0002,2,3593,5195,371
3,paragraph_based,contract_0001,context_11ef4095ae,contract_0001__chunk_0003,3,5201,7629,512
4,paragraph_based,contract_0001,context_11ef4095ae,contract_0001__chunk_0004,4,7311,7909,129


From task 10 still

In [6]:
def sha256_text(text):
    return hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()


text_by_contract = dict(
    zip(
        test_documents["contract_id"],
        test_documents["context"]
    )
)

official_test_chunks["chunk_text"] = [
    text_by_contract[contract_id][start:end]
    for contract_id, start, end in zip(
        official_test_chunks["contract_id"],
        official_test_chunks["char_start"],
        official_test_chunks["char_end"]
    )
]

official_test_chunks["chunk_hash"] = (
    official_test_chunks["chunk_text"].map(sha256_text)
)

official_test_chunks["document_hash"] = (
    official_test_chunks["contract_id"]
    .map({
        contract_id: sha256_text(text)
        for contract_id, text in text_by_contract.items()
    })
)

Save the chunks to the task13 outputs

In [7]:
OUTPUT_DIR = Path("task13_outputs") / "chunks"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

official_test_chunk_path = (
    OUTPUT_DIR / "official_test_chunks.parquet"
)

official_test_chunks.to_parquet(
    official_test_chunk_path,
    index=False
)

print("Saved to:", official_test_chunk_path)

Saved to: task13_outputs/chunks/official_test_chunks.parquet


Now we have chunked the 102 contracts from the raw test data into 2984 chunks. These chunks will now be ran through the simple classifier developed in task 12.

In [8]:
# From simple_classification.ipynb, adapted for Notebook 13

def create_embeddings(chunks):
    model = SentenceTransformer(
        "sentence-transformers/all-distilroberta-v1"
    )

    chunks = chunks.reset_index(drop=True)
    texts = chunks["chunk_text"].tolist()

    embeddings = model.encode(
        texts,
        batch_size=32,
        show_progress_bar=True,
        normalize_embeddings=True,
    )

    assert embeddings.shape == (
        len(chunks),
        model.get_embedding_dimension(),
    )

    # Save official-test embeddings separately
    output_dir = Path("task13_outputs") / "embeddings"
    output_dir.mkdir(parents=True, exist_ok=True)

    output_path = output_dir / "official_test_embeddings.npz"

    np.savez(
        output_path,
        chunk_ids=chunks["chunk_id"].to_numpy().astype(str),
        embeddings=embeddings.astype(np.float32),
    )

    print(f"Saved to: {output_path}")
    print(f"Embedding shape: {embeddings.shape}")

    return embeddings

Create the embeddings

In [9]:
from sentence_transformers import SentenceTransformer
create_embeddings(official_test_chunks)

Batches: 100%|██████████| 94/94 [00:43<00:00,  2.16it/s]

Saved to: task13_outputs/embeddings/official_test_embeddings.npz
Embedding shape: (2984, 768)


array([[ 0.00259288,  0.00538927,  0.00202559, ..., -0.03924627,
        -0.04204857,  0.04334323],
       [-0.0461052 , -0.01579425, -0.00900529, ...,  0.01655065,
        -0.03231233,  0.01272967],
       [-0.02068169, -0.02681337, -0.01830851, ..., -0.01904853,
        -0.05176236,  0.03686595],
       ...,
       [ 0.00260214,  0.0160979 , -0.03067891, ..., -0.00756703,
        -0.01839964,  0.04973701],
       [-0.00086726,  0.00566549, -0.02930059, ...,  0.03234141,
        -0.0363893 ,  0.03499537],
       [ 0.00141332,  0.01747364,  0.01157633, ...,  0.03605255,
        -0.03542272,  0.04459333]], shape=(2984, 768), dtype=float32)

In [10]:

# Load the frozen classifier from Simple Classification

FROZEN_DIR = Path("task12b_outputs")

with open(FROZEN_DIR / "logistic_baseline.pkl", "rb") as file:
    clf = pickle.load(file)

# Load the official-test embeddings created in Task 13
saved = np.load(
    "task13_outputs/embeddings/official_test_embeddings.npz"
)

official_test_embeddings = saved["embeddings"]
official_test_chunk_ids = saved["chunk_ids"].astype(str)

# Use FINAL_IDS from Simple Classification
FINAL_IDS = [
    "governing_law",
    "renewal_term",
    "revenue_profit_sharing",
    "cap_on_liability",
    "uncapped_liability",
    "termination_for_convenience",
    "anti_assignment",
    "audit_rights",
    "license_grant",
    "exclusivity",
]

# Verify category order and chunk alignment
assert list(clf.categories) == FINAL_IDS, (
    "Model categories do not match FINAL_IDS."
)

assert len(official_test_embeddings) == len(official_test_chunk_ids)
assert len(official_test_chunk_ids) == len(official_test_chunks)

assert np.array_equal(
    official_test_chunk_ids,
    official_test_chunks["chunk_id"].astype(str).to_numpy()
), "Embedding chunk IDs do not match the official-test chunks."

print("Frozen model loaded successfully.")
print("Embedding shape:", official_test_embeddings.shape)
print("Number of chunks:", len(official_test_chunk_ids))
print("Tuned threshold:", clf.threshold_)

# Generate binary predictions using the fixed predict() method
official_test_predictions = clf.predict(
    official_test_embeddings,
    threshold=0.20,
    index=official_test_chunk_ids,
)

# Verify predictions
assert official_test_predictions.shape == (
    len(official_test_chunk_ids),
    len(FINAL_IDS)
), "Unexpected prediction shape."

assert official_test_predictions.columns.tolist() == FINAL_IDS
assert official_test_predictions.index.astype(str).tolist() == (
    official_test_chunk_ids.tolist()
)
assert official_test_predictions.isin([0, 1]).all().all(), (
    "Predictions must contain only 0 and 1."
)

print(
    "Total positive predictions:",
    int(official_test_predictions.to_numpy().sum())
)

print("\nPositive predictions by category:")
print(official_test_predictions.sum())

display(official_test_predictions.head())

# Save predictions for Task 13 evaluation
OUTPUT_DIR = Path("task13_outputs/predictions")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

prediction_path = OUTPUT_DIR / "official_test_predictions.parquet"

official_test_predictions.to_parquet(
    prediction_path,
    index=True,
)

print("Predictions saved to:", prediction_path)


Frozen model loaded successfully.
Embedding shape: (2984, 768)
Number of chunks: 2984
Tuned threshold: 0.2
Total positive predictions: 777

Positive predictions by category:
governing_law                  153
renewal_term                    41
revenue_profit_sharing          65
cap_on_liability                94
uncapped_liability              20
termination_for_convenience     39
anti_assignment                115
audit_rights                    67
license_grant                  131
exclusivity                     52
dtype: int64


,governing_law,renewal_term,revenue_profit_sharing,cap_on_liability,uncapped_liability,termination_for_convenience,anti_assignment,audit_rights,license_grant,exclusivity
contract_0001__chunk_0000,0,0,0,0,0,0,0,0,0,0
contract_0001__chunk_0001,0,0,0,0,0,0,0,0,0,0
contract_0001__chunk_0002,0,0,0,0,0,0,0,0,0,0
contract_0001__chunk_0003,0,0,0,0,0,0,0,0,0,0
contract_0001__chunk_0004,0,0,0,0,0,0,0,0,0,0


Predictions saved to: task13_outputs/predictions/official_test_predictions.parquet


Verifications

In [11]:
saved_predictions = pd.read_parquet(prediction_path)

print("Shape:", saved_predictions.shape)
display(saved_predictions.head())

#check row alignment
print("Official test chunks:", len(official_test_chunks))
print("Prediction rows:", len(saved_predictions))

assert len(saved_predictions) == len(official_test_chunks), (
    "Prediction rows do not match the number of official-test chunks."
)

#check column alignment
print("Prediction columns:")
print(saved_predictions.columns.tolist())

print("\nOfficial test chunk columns:")
print(official_test_chunks.columns.tolist())

print("\nPrediction shape:", saved_predictions.shape)
print("Chunk count:", len(official_test_chunks))

print(saved_predictions.head())
print(saved_predictions.index[:5])
print(official_test_chunks["chunk_id"].head())


# verify chunk ids and prediction values
# Confirm prediction rows align with official-test chunks
prediction_ids = saved_predictions.index.astype(str)
chunk_ids = official_test_chunks["chunk_id"].astype(str).reset_index(drop=True)

assert prediction_ids.is_unique
assert chunk_ids.is_unique
assert list(prediction_ids) == list(chunk_ids), (
    "Prediction IDs and official-test chunk IDs are not aligned."
)

FINAL_IDS = [
    "governing_law",
    "renewal_term",
    "revenue_profit_sharing",
    "cap_on_liability",
    "uncapped_liability",
    "termination_for_convenience",
    "anti_assignment",
    "audit_rights",
    "license_grant",
    "exclusivity",
]

# Confirm expected categories and binary predictions
assert saved_predictions.columns.tolist() == FINAL_IDS
assert saved_predictions.isin([0, 1]).all().all()

print("PASS: Chunk IDs align.")
print("PASS: All 10 categories are present in the expected order.")
print("PASS: Predictions contain only 0 and 1.")
print("Total chunks:", len(saved_predictions))

Shape: (2984, 10)


,governing_law,renewal_term,revenue_profit_sharing,cap_on_liability,uncapped_liability,termination_for_convenience,anti_assignment,audit_rights,license_grant,exclusivity
contract_0001__chunk_0000,0,0,0,0,0,0,0,0,0,0
contract_0001__chunk_0001,0,0,0,0,0,0,0,0,0,0
contract_0001__chunk_0002,0,0,0,0,0,0,0,0,0,0
contract_0001__chunk_0003,0,0,0,0,0,0,0,0,0,0
contract_0001__chunk_0004,0,0,0,0,0,0,0,0,0,0


Official test chunks: 2984
Prediction rows: 2984
Prediction columns:
['governing_law', 'renewal_term', 'revenue_profit_sharing', 'cap_on_liability', 'uncapped_liability', 'termination_for_convenience', 'anti_assignment', 'audit_rights', 'license_grant', 'exclusivity']

Official test chunk columns:
['method', 'contract_id', 'context_group_id', 'chunk_id', 'chunk_number', 'char_start', 'char_end', 'num_tokens', 'chunk_text', 'chunk_hash', 'document_hash']

Prediction shape: (2984, 10)
Chunk count: 2984
                           governing_law  renewal_term  \
contract_0001__chunk_0000              0             0   
contract_0001__chunk_0001              0             0   
contract_0001__chunk_0002              0             0   
contract_0001__chunk_0003              0             0   
contract_0001__chunk_0004              0             0   

                           revenue_profit_sharing  cap_on_liability  \
contract_0001__chunk_0000                       0                 0   
con

In [12]:

print("ANNOTATION SETS")
print(test_annotation_sets.columns.tolist())
display(test_annotation_sets.head())

print("SPANS")
print(test_spans.columns.tolist())
display(test_spans.head())

print("CATEGORIES")
print(test_categories.columns.tolist())
display(test_categories.head())


ANNOTATION SETS
['annotation_set_id', 'contract_id', 'category_id', 'is_impossible']


,annotation_set_id,contract_id,category_id,is_impossible
0,contract_0001__affiliate_license_licensee,contract_0001,affiliate_license_licensee,True
1,contract_0001__affiliate_license_licensor,contract_0001,affiliate_license_licensor,True
2,contract_0001__agreement_date,contract_0001,agreement_date,True
3,contract_0001__anti_assignment,contract_0001,anti_assignment,True
4,contract_0001__audit_rights,contract_0001,audit_rights,True


SPANS
['span_id', 'annotation_set_id', 'source_qa_id', 'answer_text', 'answer_start', 'answer_end']


,span_id,annotation_set_id,source_qa_id,answer_text,answer_start,answer_end
0,contract_0001__document_name__span_000,contract_0001__document_name,LohaCompanyltd_20191209_F-1_EX-10.16_11917878_...,SUPPLY CONTRACT,14,29
1,contract_0001__parties__span_000,contract_0001__parties,LohaCompanyltd_20191209_F-1_EX-10.16_11917878_...,The seller:,143,154
2,contract_0001__parties__span_001,contract_0001__parties,LohaCompanyltd_20191209_F-1_EX-10.16_11917878_...,The buyer/End-User: Shenzhen LOHAS Supply Chai...,49,117
3,contract_0001__expiration_date__span_000,contract_0001__expiration_date,LohaCompanyltd_20191209_F-1_EX-10.16_11917878_...,"The Contract is valid for 5 years, beginning f...",10985,11049
4,contract_0001__governing_law__span_000,contract_0001__governing_law,LohaCompanyltd_20191209_F-1_EX-10.16_11917878_...,It will be governed by the law of the People's...,10691,10859


CATEGORIES
['category_id', 'category_name', 'question']


,category_id,category_name,question
0,document_name,Document Name,Highlight the parts (if any) of this contract ...
1,parties,Parties,Highlight the parts (if any) of this contract ...
2,agreement_date,Agreement Date,Highlight the parts (if any) of this contract ...
3,effective_date,Effective Date,Highlight the parts (if any) of this contract ...
4,expiration_date,Expiration Date,Highlight the parts (if any) of this contract ...


# Explore Metrics
first we need to find the ground-truth labels

In [13]:
# From simple_classification.ipynb
def find_positives(chunks, spans):
    """
    Find chunks that fully contain annotated answer spans.
    Returns unique (category_id, chunk_id) pairs.
    """
    chunks_by_contract_idx = chunks.groupby("contract_id").indices
    start = chunks["char_start"].to_numpy()
    end = chunks["char_end"].to_numpy()

    hits = []
    pos_found = 0

    for row in spans.itertuples():
        chunk_indices = chunks_by_contract_idx.get(row.contract_id)

        if chunk_indices is None:
            continue

        matches = chunk_indices[
            (start[chunk_indices] <= row.answer_start)
            & (row.answer_end <= end[chunk_indices])
        ]

        pos_found += len(matches) > 0
        hits.extend(
            (row.category_id, chunks["chunk_id"].iat[i])
            for i in matches
        )

    positives = pd.DataFrame(
        hits, columns=["category_id", "chunk_id"]
    ).drop_duplicates()

    return positives, pos_found


Find positives, which spans match a chunk

In [14]:

# Select the official-test annotation sets for the final 10 categories
official_test_sets = test_annotation_sets[
    test_annotation_sets["category_id"].isin(FINAL_IDS)
].copy()

# Keep spans associated with those annotation sets
official_test_spans = test_spans.merge(
    official_test_sets[
        ["annotation_set_id", "category_id", "contract_id"]
    ],
    on="annotation_set_id",
    how="inner",
    validate="many_to_one",
)

official_test_positives, pos_found = find_positives(
    official_test_chunks,
    official_test_spans,
)

print(
    f"{pos_found}/{len(official_test_spans)} "
    "official-test spans matched a chunk"
)

display(
    official_test_positives.groupby("category_id")
    .size()
    .reindex(FINAL_IDS, fill_value=0)
    .rename("positive_chunks")
    .to_frame()
)


818/838 official-test spans matched a chunk


,positive_chunks
category_id,
governing_law,89
renewal_term,26
revenue_profit_sharing,55
cap_on_liability,81
uncapped_liability,14
termination_for_convenience,35
anti_assignment,101
audit_rights,66
license_grant,96


Create y_true, hard truths

In [15]:
#from simple_classification.ipnyb
def get_chunk_categories(chunks, positives, categories=FINAL_IDS):
    """Create one row for every chunk/category combination."""
    chunk_categories = pd.merge(
        chunks[["chunk_id"]],
        pd.Series(categories, name="category_id"),
        how="cross",
    )

    chunk_categories = chunk_categories.merge(
        positives.assign(is_contained=True),
        on=["chunk_id", "category_id"],
        how="left",
    )

    chunk_categories["is_contained"] = (
        chunk_categories["is_contained"]
        .fillna(False)
        .astype(bool)
    )

    return chunk_categories

###

official_test_chunk_categories = get_chunk_categories(
    official_test_chunks,
    official_test_positives,
)

# Convert the long table to a chunk-by-category matrix
y_true = (
    official_test_chunk_categories
    .pivot(
        index="chunk_id",
        columns="category_id",
        values="is_contained",
    )
    .reindex(
        index=official_test_chunks["chunk_id"],
        columns=FINAL_IDS,
    )
    .fillna(False)
    .astype(int)
)

# Align the prediction matrix to the exact same chunk order
y_pred = saved_predictions.copy()
y_pred.index = y_pred.index.astype(str)
y_true.index = y_true.index.astype(str)
y_pred = y_pred.reindex(index=y_true.index, columns=FINAL_IDS)

assert y_true.index.equals(y_pred.index)
assert list(y_true.columns) == list(y_pred.columns) == FINAL_IDS
assert y_true.shape == (len(official_test_chunks), len(FINAL_IDS))

print("Ground truth shape:", y_true.shape)
print("Prediction shape:", y_pred.shape)
print("Ground-truth positive counts:")
display(y_true.sum().rename("positive_chunks").to_frame())


# Check whether true labels and predictions contain positive values
print("Total true positives:", int(y_true.to_numpy().sum()))
print("Total predicted positives:", int(y_pred.to_numpy().sum()))

# Check each category
for cat in y_true.columns:
    true_pos = int(y_true[cat].sum())
    pred_pos = int(y_pred[cat].sum())
    overlap = int(((y_true[cat] == 1) & (y_pred[cat] == 1)).sum())

    print(
        f"{cat}: "
        f"true={true_pos}, "
        f"predicted={pred_pos}, "
        f"overlap={overlap}"
    )



Ground truth shape: (2984, 10)
Prediction shape: (2984, 10)
Ground-truth positive counts:


,positive_chunks
category_id,
governing_law,89
renewal_term,26
revenue_profit_sharing,55
cap_on_liability,81
uncapped_liability,14
termination_for_convenience,35
anti_assignment,101
audit_rights,66
license_grant,96


Total true positives: 624
Total predicted positives: 777
governing_law: true=89, predicted=153, overlap=69
renewal_term: true=26, predicted=41, overlap=16
revenue_profit_sharing: true=55, predicted=65, overlap=31
cap_on_liability: true=81, predicted=94, overlap=47
uncapped_liability: true=14, predicted=20, overlap=10
termination_for_convenience: true=35, predicted=39, overlap=12
anti_assignment: true=101, predicted=115, overlap=64
audit_rights: true=66, predicted=67, overlap=42
license_grant: true=96, predicted=131, overlap=63
exclusivity: true=61, predicted=52, overlap=18


calculates category-level precision, recall, F1, and support

In [16]:

from sklearn.metrics import precision_score, recall_score, f1_score

#from simple_classification.ipynb
def evaluate(y_true, y_pred):
    """Calculate metrics for each category."""
    rows = []

    for cat in y_true.columns:
        true_values = y_true[cat].to_numpy()
        pred_values = y_pred[cat].to_numpy()

        rows.append({
            "category": cat,
            "positive_support": int(true_values.sum()),
            "precision": precision_score(
                true_values, pred_values, zero_division=0
            ),
            "recall": recall_score(
                true_values, pred_values, zero_division=0
            ),
            "f1": f1_score(
                true_values, pred_values, zero_division=0
            ),
        })

    return pd.DataFrame(rows).set_index("category")


category_metrics = evaluate(y_true, y_pred)
display(category_metrics.round(4))


,positive_support,precision,recall,f1
category,,,,
governing_law,89,0.4510,0.7753,0.5702
renewal_term,26,0.3902,0.6154,0.4776
revenue_profit_sharing,55,0.4769,0.5636,0.5167
cap_on_liability,81,0.5000,0.5802,0.5371
uncapped_liability,14,0.5000,0.7143,0.5882
termination_for_convenience,35,0.3077,0.3429,0.3243
anti_assignment,101,0.5565,0.6337,0.5926
audit_rights,66,0.6269,0.6364,0.6316
license_grant,96,0.4809,0.6562,0.5551


calculate macro and micro averages

In [17]:

from sklearn.metrics import precision_recall_fscore_support

# Macro: calculate each category's score, then average the 10 scores
macro_metrics = {
    "precision": category_metrics["precision"].mean(),
    "recall": category_metrics["recall"].mean(),
    "f1": category_metrics["f1"].mean(),
}

# Micro: combine all chunk/category decisions before scoring
micro_p, micro_r, micro_f1, _ = precision_recall_fscore_support(
    y_true.to_numpy().ravel(),
    y_pred.to_numpy().ravel(),
    average="binary",
    zero_division=0,
)

overall_metrics = pd.DataFrame(
    [
        {"average": "macro", **macro_metrics},
        {
            "average": "micro",
            "precision": micro_p,
            "recall": micro_r,
            "f1": micro_f1,
        },
    ]
).set_index("average")

display(overall_metrics.round(4))


,precision,recall,f1
average,,,
macro,0.4636,0.5813,0.5112
micro,0.4788,0.5962,0.5310


Save outputs

In [18]:

METRICS_DIR = Path("task13_outputs/metrics")
METRICS_DIR.mkdir(parents=True, exist_ok=True)

category_metrics.to_csv(METRICS_DIR / "category_metrics.csv")
overall_metrics.to_csv(METRICS_DIR / "overall_metrics.csv")
y_true.to_parquet(METRICS_DIR / "official_test_y_true.parquet")

print("Saved Task 13 metrics to:", METRICS_DIR)


Saved Task 13 metrics to: task13_outputs/metrics


# Comparison to results in simple_classification.ipynb

Task 13's macro-F1 decreased from 0.5350 on validation to 0.5112 on the official test set, a drop of 0.0238. Recall remained almost unchanged, while precision decreased by 0.0414. This suggests that the model identifies a similar proportion of relevant clauses but produces more false-positive predictions on the official test set.

Important distinction: Task 12's (simple_classification) validation results were used during model selection, while Task 13's official test results provide a more independent assessment of generalization.

In [19]:
# Task 12 validation metrics
task12_metrics = {
    "Macro precision": 0.5050,
    "Macro recall": 0.5820,
    "Macro-F1": 0.5350,
}

# Task 13 official test metrics
task13_metrics = {
    "Macro precision": 0.4636,
    "Macro recall": 0.5813,
    "Macro-F1": 0.5112,
    "Micro precision": 0.4788,
    "Micro recall": 0.5962,
    "Micro-F1": 0.5310,
}

comparison = pd.DataFrame({
    "Task 12 (Validation)": task12_metrics,
    "Task 13 (Official Test)": task13_metrics,
})

comparison["Change"] = (
    comparison["Task 13 (Official Test)"]
    - comparison["Task 12 (Validation)"]
)

comparison.index.name = "Metric"

print(comparison.round(4).to_string())

                 Task 12 (Validation)  Task 13 (Official Test)  Change
Metric                                                                
Macro precision                 0.505                   0.4636 -0.0414
Macro recall                    0.582                   0.5813 -0.0007
Macro-F1                        0.535                   0.5112 -0.0238
Micro precision                   NaN                   0.4788     NaN
Micro recall                      NaN                   0.5962     NaN
Micro-F1                          NaN                   0.5310     NaN
